# Notebook 08 — Metastasis class: model predictions and image features per patient (CPU is enough)
Exports two small files for the metastasis-class analysis (no images, only patient IDs and numbers):
1. `mets_predictions.csv` — for every metastatic patient, the probability of "metastatic" from Experiment 1 (own-machine model,
   out-of-fold), averaged over the 4 backbones and 3 seeds, for T2 (metastatic vs non-metastatic) and T3 (metastatic vs normal),
   ImageNet and BS-80K initialisation. Used to check which patients the model misses.
2. `mets_features.npz` — image features of every metastatic patient from the BS-80K-trained EfficientNetV2-S
   (anterior + posterior, 2 x 1280 numbers). Used to see which groups a scan model forms by itself.

**Runtime:** CPU is enough (Runtime > Change runtime type > CPU). Run all, ~10-20 min.

In [ ]:
# settings and data
import os, sys, subprocess, json
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'timm'], check=False)
import numpy as np, pandas as pd, torch, timm
from PIL import Image
LOCAL_TEST = bool(os.environ.get('BONESCAN_BASE'))
if LOCAL_TEST:
    BASE = os.environ['BONESCAN_BASE']
else:
    from google.colab import drive
    drive.mount('/content/drive')
    BASE = '/content/drive/MyDrive/Colab Notebooks/bonescan'
EXP12, BSD, P3 = BASE + '/revision/machine_v3', BASE + '/bs80k', BASE + '/prepared_v3'
OUT = BASE + '/revision/mets_analysis'
os.makedirs(OUT, exist_ok=True)
MODELS = ['tf_efficientnetv2_s', 'resnext50_32x4d', 'regnety_016', 'inception_v3']
TASKS = {'T2': 'cancer_vs_noncancer', 'T3': 'sec_vs_normal'}
SEEDS, INITS, H, W = [42, 43, 44], ['IN', 'BS'], 512, 176
allp = pd.read_csv(P3 + '/patients.csv')
pool = allp[allp.split == 'pool'].copy(); pool['fold'] = pool.fold.astype(int)
# patients relabelled after training (IDs kept locally in relabelled_ids.json, not published)
RELABELLED = set(json.load(open(BASE + '/relabelled_ids.json'))) if os.path.exists(BASE + '/relabelled_ids.json') else set()
mets = pool[(pool.label == 'SECONDARY') & (~pool.patient_id.str.lower().isin(RELABELLED))].reset_index(drop=True)
FOLD = dict(zip(pool.patient_id, pool.fold))
print('metastatic patients:', len(mets), mets.machine.value_counts().to_dict())

In [ ]:
# 1. per-patient predictions from Experiment 1 (out-of-fold, own machine)
rows = []
for tk, task in TASKS.items():
    for init in INITS:
        for cam in ['M1', 'M2']:
            P = []
            for name in MODELS:
                for s in SEEDS:
                    for k in range(5):
                        f = f'{EXP12}/preds/mach_{task}_{name}_{init}_{cam}_s{s}_fold{k}_oof.csv'
                        if os.path.exists(f):
                            x = pd.read_csv(f); x = x[x.patient_id.map(FOLD) == k]
                            P.append(x[['patient_id', 'prob_1']].assign(model=name, seed=s))
            if not P:
                continue
            P = pd.concat(P)
            g = P.groupby('patient_id').prob_1.agg(['mean', 'count']).reset_index()
            g = g[g.patient_id.isin(set(mets.patient_id))]
            rows.append(g.assign(task=tk, init=init, machine=cam))
pred = pd.concat(rows).rename(columns={'mean': 'p_metastatic', 'count': 'n_models_x_seeds'})
pred['detected'] = (pred.p_metastatic > 0.5).astype(int)
pred.to_csv(OUT + '/mets_predictions.csv', index=False)
print(pred.groupby(['task', 'init']).agg(patients=('patient_id', 'nunique'), detected=('detected', 'mean'), models_x_seeds=('n_models_x_seeds', 'median')))

In [ ]:
# 2. image features from the BS-80K-trained EfficientNetV2-S
fix = lambda p: P3 + '/images/' + os.path.basename(str(p))
bb = timm.create_model('tf_efficientnetv2_s', pretrained=False, num_classes=0)
sd = torch.load(f'{BSD}/model_bs80k_tf_efficientnetv2_s.pt', map_location='cpu')
res = bb.load_state_dict({k[len('backbone.'):]: v for k, v in sd.items() if k.startswith('backbone.')}, strict=False)
assert not res.missing_keys, res.missing_keys[:3]
bb.eval()
MEAN = torch.tensor([0.485, 0.456, 0.406]).view(1, 3, 1, 1); STD = torch.tensor([0.229, 0.224, 0.225]).view(1, 3, 1, 1)
def img(p):
    a = torch.from_numpy(np.array(Image.open(fix(p)).convert('L'))).float() / 255.0
    return ((a.view(1, 1, H, W).repeat(1, 3, 1, 1)) - MEAN) / STD
Z = []
with torch.no_grad():
    for a, p in zip(mets.ANT, mets.POST):
        Z.append(torch.cat([bb(img(a)), bb(img(p))], 1).numpy()[0])
Z = np.stack(Z).astype(np.float32)
np.savez_compressed(OUT + '/mets_features.npz', ids=mets.patient_id.values, machine=mets.machine.values, z=Z)
print('features', Z.shape, '| saved to', OUT)
print('DONE')